# Diffusion World Model Steering — Dubins Car Demo

Run from the `release/` directory (parent of `dubins/`).

In [ ]:
import os, sys
import types
import numpy as np
import torch
import yaml
import imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display

# Ensure cwd is release/ so config relative paths work
if os.path.basename(os.getcwd()) == 'dubins':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from dubins.steer import load_steering_model, rollout_no_grad, optimize_noise, decode_latents_to_images
from dubins.env import DubinsConfig, simulate_trajectory

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Load model

In [ ]:
with open('dubins/config.yaml') as f:
    config = yaml.safe_load(f)

with open('dubins/steering_config.yaml') as f:
    steering_cfg = yaml.safe_load(f)

ckpts = config['checkpoints']
model = load_steering_model(ckpts['world_model'], ckpts['vae'], config, device)
print('Model loaded.')

## Example trajectory — initial frame

In [ ]:
dubins_cfg = DubinsConfig()
ph = config.get('prediction_heads', {})
dubins_cfg.obs_x = ph.get('obs_x', 0.0)
dubins_cfg.obs_y = ph.get('obs_y', 0.0)
dubins_cfg.obs_r = ph.get('obs_r', 0.25)

init_state = torch.from_numpy(np.load('dubins/example_traj/init_state.npy')).float()
actions    = torch.from_numpy(np.load('dubins/example_traj/actions.npy')).float()

gt_states, gt_images = simulate_trajectory(init_state, actions, dubins_cfg)

plt.figure(figsize=(3, 3))
plt.imshow(gt_images[0])
plt.title('Initial frame')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'init_state: {init_state.tolist()}   traj_length: {len(actions)}')

In [ ]:
# Encode initial frame and build rollout tensors
num_steps_cond = config['denoiser']['num_steps_conditioning']
init_img_t = torch.from_numpy(gt_images[0]).float().to(device).permute(2, 0, 1) / 127.5 - 1.0

with torch.no_grad():
    z_init = model.encode_images(init_img_t.unsqueeze(0), sample=False)
    z_init = z_init.unsqueeze(0).repeat(1, num_steps_cond, 1, 1, 1)

historical_actions = torch.zeros(1, num_steps_cond - 1, 1, device=device)
future_actions     = actions.unsqueeze(0).unsqueeze(-1).to(device)
rollout_actions    = torch.cat([historical_actions, future_actions], dim=1)

latent_shape = z_init.shape[2:]
T_future     = rollout_actions.shape[1] - (num_steps_cond - 1)
init_frame   = np.array(gt_images[0])[np.newaxis]  # (1, H, W, 3)

os.makedirs('steering_results', exist_ok=True)

In [ ]:
def save_video(frames_list, path, fps=10):
    """Concatenate a list of (T, H, W, 3) arrays horizontally and save as mp4."""
    T = min(len(f) for f in frames_list)
    writer = imageio.get_writer(path, fps=fps, codec='libx264', bitrate='8M')
    for t in range(T):
        frame = np.concatenate([f[t] for f in frames_list], axis=1)
        writer.append_data(frame)
    writer.close()

def show_video(path, width=900):
    display(Video(path, embed=True, width=width))

## Nominal rollouts (5 samples)

In [ ]:
nominal_clips = []
for seed in range(5):
    torch.manual_seed(seed)
    noise = torch.randn(T_future, *latent_shape, device=device)
    with torch.no_grad():
        latents = rollout_no_grad(model, z_init, rollout_actions, noise, num_steps_cond)
        frames  = decode_latents_to_images(model, latents)
    nominal_clips.append(np.concatenate([init_frame, frames], axis=0))

save_video(nominal_clips, 'steering_results/nominal.mp4')
show_video('steering_results/nominal.mp4')

## Pessimistic steering (toward failure set)

In [ ]:
def make_args(**overrides):
    args = types.SimpleNamespace(**steering_cfg, seed=42)
    for k, v in overrides.items():
        setattr(args, k, v)
    return args

args = make_args(mode='pessimistic', margin_coeff=20.0)
opt_latents, _, _, _ = optimize_noise(model, z_init, rollout_actions, config, device, args)

opt_frames = decode_latents_to_images(model, opt_latents.to(device))
save_video([np.concatenate([init_frame, opt_frames], axis=0)], 'steering_results/pessimistic.mp4')
show_video('steering_results/pessimistic.mp4', width=300)

## Optimistic steering (toward safety)

In [ ]:
args = make_args(mode='optimistic')
opt_latents, _, _, _ = optimize_noise(model, z_init, rollout_actions, config, device, args)

opt_frames = decode_latents_to_images(model, opt_latents.to(device))
save_video([np.concatenate([init_frame, opt_frames], axis=0)], 'steering_results/optimistic.mp4')
show_video('steering_results/optimistic.mp4', width=300)

## Pessimistic steering — no typical set regularizer

In [ ]:
args = make_args(
    mode='pessimistic', margin_coeff=20.0,
    kl_coeff=0.0, kl_coeff_spherical=0.0,
    std_coeff=0.0, spectral_coeff=0.0,
    std_permutation_coeff=0.0, margin_range_coeff=0.0,
    normalize_noise=False,
)
opt_latents, _, _, _ = optimize_noise(model, z_init, rollout_actions, config, device, args)

opt_frames = decode_latents_to_images(model, opt_latents.to(device))
save_video([np.concatenate([init_frame, opt_frames], axis=0)], 'steering_results/pessimistic_noreg.mp4')
show_video('steering_results/pessimistic_noreg.mp4', width=300)